# Audited Superstore analysis

This notebook now uses the same validated, Streamlit-independent calculations as
the dashboard. A **transaction** is one dataframe row; an **order** is a distinct
`Order ID`. All financial figures below are historical observations from the
supplied **2011–2014** data. They are not forecasts, annualized impacts, recovered
profit, or realized savings.

See [the analytical methodology](docs/METHODOLOGY.md) for metric definitions.
Run `python scripts/report_findings.py` from the repository root to reproduce the
current full-data findings.

# Part 1. DATA CLEANING AND EXPLORATION

### Step 1: Use the Repository Dataset

In [ ]:
# Run this notebook from the repository root so Superstore.csv and src/ resolve.
# No upload step or external service is required.

### Step 2. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
print("Libraries imported successfully")

### Step 3: Load and Validate the Data

In [ ]:
from pathlib import Path

from src.data import validate_data

DATA_PATH = Path("Superstore.csv")
raw_df = pd.read_csv(DATA_PATH, encoding="latin-1")
raw_df["Order Date"] = pd.to_datetime(raw_df["Order Date"], format="%d-%m-%Y")
raw_df["Ship Date"] = pd.to_datetime(raw_df["Ship Date"], format="%d-%m-%Y")
df = validate_data(raw_df)

print("Dataset loaded and validated successfully")
print(f"Shape: {df.shape[0]:,} transactions, {df.shape[1]} validated columns")
print(f"Date range: {df['Order Date'].min():%Y-%m-%d} to {df['Order Date'].max():%Y-%m-%d}")

The supplied dataset contains **9,994 transactions**, **5,009 unique orders**, and validated dates from **2011-01-04 through 2014-12-31**.

### Step 4: Initial Data Exploration

In [ ]:
print("="*80)
print("FIRST 5 ROWS")
print("="*80)
display(df.head())

print("\n" + "="*80)
print("DATASET INFORMATION")
print("="*80)
df.info()

print("\n" + "="*80)
print("STATISTICAL SUMMARY")
print("="*80)
display(df.describe())

print("\n" + "="*80)
print("COLUMN NAMES")
print("="*80)
display(df.columns.tolist())



The above code cell is used to print first 5 rows, summary of the data, info of the data, and column names in super store sales dataset.

### Step 5: Check Data Quality

In [ ]:
print("\n" + "="*80)
print("MISSING VALUES")
print("=" * 80)
missing = df.isnull().sum()
missing_pct = df.isnull().sum() / len(df) * 100
missing_df = pd.DataFrame({'Column':missing.index,
                           'Missing Count':missing.values,
                           'Percentage':missing_pct.values})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values(by = "Missing Count", ascending = False)
print(missing_df if len(missing_df) > 0 else "✅ No missing values found!")

# Check for duplicates
print("\n"+ "=" * 80)
print("DUPLICATE ROWS")
print("=" * 80)
duplicates = df.duplicated().sum()
print(f"NUmber of duplicate rows: {duplicates}" )


# Check unique value for key categorical columns
print("\n" + "=" * 80)
print("UNIQUE VALUES IN CATEGORICAL COLUMNS")
print("=" * 80)
categorical_cols = df.select_dtypes(include = ["object"]).columns
for col in categorical_cols:
    print(f"{col}: {df[col].nunique()} unique values")
    if df[col].nunique() < 20:
        print(f"Values: {df[col].unique()}")
    print()

The output showed that there is no missing value or duplicate rows in the dataset.

### Step 6: Initial Business Questions

In [ ]:
from src.analytics import calculate_kpis

kpis = calculate_kpis(df)
print("QUICK BUSINESS INSIGHTS")
print(f"Total Sales: ${kpis['sales']:,.2f}")
print(f"Total Profit: ${kpis['profit']:,.2f}")
print(f"Unique Orders: {kpis['unique_order_count']:,}")
print(f"Transactions: {kpis['transaction_count']:,}")
print(f"Customers: {df['Customer ID'].nunique():,}")
print(f"Profit Margin: {kpis['profit_margin']:.2f}%")
print(f"Loss-making Transactions: {kpis['unprofitable_transaction_rate']:.2f}%")

### Step 7: Verify Parsed Date Columns

In [ ]:
print("DATE VALIDATION")
print(f"Order Date Range: {df['Order Date'].min():%Y-%m-%d} to {df['Order Date'].max():%Y-%m-%d}")
print(f"Ship Date Range: {df['Ship Date'].min():%Y-%m-%d} to {df['Ship Date'].max():%Y-%m-%d}")
display(df[["Order Date", "Ship Date"]].dtypes)

### Step 8: Create Time-Based Features

In [ ]:
df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month
df["Month Name"] = df["Order Date"].dt.month_name()
df["Quarter"] = df["Order Date"].dt.quarter
df["Day of Week"] = df["Order Date"].dt.day_name()
df["Week of Year"] = df["Order Date"].dt.isocalendar().week
df["Days to Ship"] = (df["Ship Date"] - df["Order Date"]).dt.days

print("Time fields created")
print(df["Year"].value_counts().sort_index())
print(f"Average days to ship: {df['Days to Ship'].mean():.2f}")

### Step 9: Create Financial Metrics

In [ ]:
# Derived descriptive fields used only for notebook exploration.
df["Profit Margin %"] = np.where(df["Sales"] != 0, df["Profit"] / df["Sales"] * 100, 0.0)
df["Is Profitable"] = df["Profit"] > 0
df["Is Loss Making"] = df["Profit"] < 0

print("Derived fields created: Profit Margin %, Is Profitable, Is Loss Making")
print("Dashboard aggregate margins are recomputed as sum(Profit) / sum(Sales), not mean row margins.")
print("No discount-dollar amount or counterfactual list price is inferred from Sales.")

### Step 10: Create Customer & Product Aggregations

In [ ]:
# Customer-level observed metrics. These are selected-period values, not forecasts.
customer_metrics = df.groupby("Customer ID").agg(
    Customer_Name=("Customer Name", "first"),
    Selected_Period_Sales=("Sales", "sum"),
    Selected_Period_Profit=("Profit", "sum"),
    Transaction_Count=("Order ID", "size"),
    Unique_Order_Count=("Order ID", "nunique"),
).reset_index()

print(f"Customers: {len(customer_metrics):,}")
display(customer_metrics.sort_values("Selected_Period_Profit", ascending=False).head())

### Step 11: Creating Category Performance Metrics

In [ ]:
from src.analytics import calculate_performance

category_performance = calculate_performance(df, "Category")
subcategory_performance = calculate_performance(df, "Sub-Category")

print("CATEGORY PERFORMANCE")
display(category_performance)
print("SUB-CATEGORY PERFORMANCE")
display(subcategory_performance.sort_values("Profit", ascending=False))

### Step 12: Analyze Discount Impact

In [ ]:
from src.analytics import calculate_discount_profitability, calculate_discount_segments

discount_analysis = calculate_discount_profitability(df)
discount_groups = calculate_discount_segments(df)

print("DISCOUNT IMPACT ANALYSIS")
print("Profitable transaction rate = count(Profit > 0) / transaction count × 100.")
display(discount_analysis)
display(discount_groups)

### Step 13: Regional & Segment Analysis

In [ ]:
from src.analytics import calculate_performance, calculate_region_performance

region_performance = calculate_region_performance(df)
segment_performance = calculate_performance(df, "Segment")

print("REGIONAL PERFORMANCE")
display(region_performance)
print("CUSTOMER-SEGMENT PERFORMANCE")
display(segment_performance)

### Step 14: Summary of Data Preparation

In [ ]:
# Final data check
print("\n" + "=" * 80)
print("DATA PREPARATION COMPLETE ✅")
print("=" * 80)

print(f"Original columns: 21")
print(f"New columns added: {len(df.columns) - 21}")
print(f"Total columns now: {len(df.columns)}")

print("\n📋 All columns:")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

print(f"\n✅ Dataset shape: {df.shape}")
print("✅ Ready for visualization and dashboard creation!")

### Step 15: Save Cleaned Data

In [ ]:
# The dashboard validates the supplied repository artifact at load time.
# Export a fresh validated source-column copy only when explicitly needed:
# df.to_csv("superstore_cleaned.csv", index=False)
print("Validated data is ready. No file was overwritten by this notebook run.")

---

## Data preparation summary

- **9,994 transactions** and **5,009 unique orders** from **2011–2014**
- **793 customers**
- Total sales: **$2,297,200.86**
- Total profit: **$286,397.02**
- Aggregate profit margin: **12.47%**
- **18.72%** of transactions were loss-making (`Profit < 0`)

These figures are descriptive results for the supplied sample. They do not
represent an identified retailer's realized impact or a causal policy result.

# Part 2: Audited Insights and Visualizations

## Discount profitability by exact observed discount

In [ ]:
from src.analytics import calculate_discount_profitability

profit_cliff = calculate_discount_profitability(df)
display(profit_cliff[[
    "Discount", "transaction_count", "profitable_transaction_count",
    "unprofitable_transaction_count", "break_even_transaction_count",
    "profitable_transaction_rate", "transaction_share",
]])

In [ ]:
from src.charts import discount_profitability_chart

fig = discount_profitability_chart(profit_cliff)
fig.show()

### Finding 2: Observed High-Discount Loss Exposure

In [ ]:
from src.analytics import DISCOUNT_RISK_THRESHOLD, calculate_discount_loss_exposure

exposure = calculate_discount_loss_exposure(df)
print(f"Threshold: Discount >= {DISCOUNT_RISK_THRESHOLD:.0%}")
print(f"Transactions: {exposure['transaction_count']:,}")
print(f"Loss-making transactions: {exposure['unprofitable_transaction_count']:,}")
print(f"Loss-making transaction rate: {exposure['unprofitable_transaction_rate']:.2f}%")
print(f"Observed gross loss exposure: ${exposure['gross_loss_exposure']:,.2f}")
print(f"Net profit across all transactions in the group: ${exposure['net_profit']:,.2f}")
print("Historical amounts only; neither amount is recoverable savings or an annual forecast.")

In [ ]:
from src.charts import discount_scatter

fig = discount_scatter(df, DISCOUNT_RISK_THRESHOLD)
fig.show()

### Finding 3: Customer Profit Concentration

In [ ]:
from src.analytics import calculate_customer_value_segments

customers, customer_segments = calculate_customer_value_segments(df)
display(customer_segments)
display(customers.head(10))

top = customer_segments.iloc[0]
print(
    f"The highest-profit {int(top['customer_count']):,} of {len(customers):,} customers "
    f"contributed ${top['Profit']:,.2f}, or {top['Net Profit Share %']:.2f}% of total net profit."
)
print("Negative-profit customers can make a segment share exceed 100%.")
print("These are observed selected-period values, not customer lifetime forecasts.")

### Finding 4: Product Profitability

In [ ]:
from src.analytics import calculate_performance

product_performance = calculate_performance(df, "Sub-Category").sort_values("Profit")
display(product_performance)
loss_making_subcategories = product_performance[product_performance["Profit"] < 0]
print(f"{len(loss_making_subcategories)} of {len(product_performance)} sub-categories have negative net profit.")
print("The dataset does not establish that loss-making products cause later purchases or preserve future revenue.")

### Finding 5: Quarterly Performance

In [ ]:
from src.analytics import calculate_quarterly_performance

quarterly_analysis = calculate_quarterly_performance(df)
display(quarterly_analysis)
q4 = quarterly_analysis.loc[quarterly_analysis["Quarter"] == 4].iloc[0]
print(
    f"Q4 sales were ${q4['Sales']:,.2f}, with net profit of ${q4['Profit']:,.2f} "
    f"and a {q4['Profit Margin %']:.2f}% aggregate margin."
)
print("No future discount-policy impact is estimated because price and volume responses are not observed.")